In [1]:
 
#Loading in Packages and Data

#Importing Packages
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import ScalarFormatter
import matplotlib.gridspec as gridspec
import xarray as xr
import os; import time
import pickle
import h5py
###############################################################
def coefs(coefficients,degree):
    coef=coefficients
    coefs=""
    for n in range(degree, -1, -1):
        string=f"({coefficients[len(coef)-(n+1)]:.1e})"
        coefs+=string + f"x^{n}"
        if n != 0:
            coefs+=" + "
    return coefs
###############################################################

#Importing Model Data
check=False
dir='/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'
# data=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_test7tundra-7_062217.nc') 
# parcel=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_pdata_test5tundra-7_062217.nc')
res='1km'
Np_str='125e3'

# Np_str='125e3'
data=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_1km_1e6.nc') #***
parcel=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_pdata_1km_1e6.nc') #***
res='1km'
Np_str='1e6'


true_time=data['time']
times=data['time'].values/(1e9 * 60); times=times.astype(float);


# #uncomment if using 250m data
# #Importing Model Data
# check=False
# dir2='/home/air673/koa_scratch/'
# data=xr.open_dataset(dir2+'cm1out_250m.nc') #***
# parcel=xr.open_dataset(dir2+'cm1out_pdata_250m.nc') #***

# # Restricts the timesteps of the data from timesteps0 to 140
# data=data.isel(time=np.arange(0,400+1))
# parcel=parcel.isel(time=np.arange(0,400+1))

In [2]:
def check_memory():
    import sys
    ipython_vars = ["In", "Out", "exit", "quit", "get_ipython", "ipython_vars"]
    print("Top 10 objects with highest memory usage")
    # Get a sorted list of the objects and their sizes
    mem = {
        key: round(value/1e6,2)
        for key, value in sorted(
            [
                (x, sys.getsizeof(globals().get(x)))
                for x in globals()
                if not x.startswith("_") and x not in sys.modules and x not in ipython_vars
            ],
            key=lambda x: x[1],
            reverse=True)[:10]
    }
    print({key:f"{value} MB" for key,value in mem.items()})
    print(f"\n{round(sum(mem.values()),2)/1000} GB in use overall")

In [3]:
###########################################################################################################################################################################

In [4]:
#LOADING VARIABLES
###############################################################

In [5]:
# Reading Back Data Later
##############
import h5py
dir2=dir+'Project_Algorithms/Lagrangian_Arrays/'
open_file=dir2+f'lagrangian_binary_array_{res}_{Np_str}_5min.h5'
with h5py.File(open_file, 'r') as f:
    Z = f['Z'][:]
    Y = f['Y'][:]
    X = f['X'][:]

# #Making Time Matrix
# rows, cols = A.shape[0], A.shape[1]
# T = np.arange(rows).reshape(-1, 1) * np.ones((1, cols), dtype=int)
check_memory()

Top 10 objects with highest memory usage
{'Z': '1064.0 MB', 'Y': '1064.0 MB', 'X': '1064.0 MB', 'NamespaceMagics': '0.0 MB', 'Normalize': '0.0 MB', 'MaxNLocator': '0.0 MB', 'ScalarFormatter': '0.0 MB', 'times': '0.0 MB', 'open_file': '0.0 MB', 'open': '0.0 MB'}

3.192 GB in use overall


In [6]:
def call_variable(varname):
    if varname=='th_e':
        with h5py.File(dir + 'Variable_Calculation/' + 'theta_e'+f'_{res}_{Np_str}'+'.h5', 'r') as f:
            var_data = f['theta_e'][:]
    elif varname=='HMC':
        dir2='/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'
        file_path = dir2 + 'Variable_Calculation/' + '2D_Moisture_Convergence' + f'_{res}_{Np_str}' + '.h5'
        with h5py.File(file_path, 'r') as f:
            var_data = f['conv'][:]
    elif varname=='VMF_c':
        dir2 = '/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'
        dir3 = dir2 + 'Project_Algorithms/Entrainment/'
        with h5py.File(dir3 + '3D_Eulerian_VMF'+f'_{res}_{Np_str}'+'.h5', 'r') as f:
            var_data = f['VMF_c'][:]
    elif varname=='VMF_g':
        dir2 = '/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'
        dir3 = dir2 + 'Project_Algorithms/Entrainment/'
        with h5py.File(dir3 + '3D_Eulerian_VMF'+f'_{res}_{Np_str}'+'.h5', 'r') as f:
            var_data = f['VMF_g'][:]
    else:
        var_data=data[varname].data
    return var_data

In [7]:
def make_lagrangian_array(varname):
    var_data=call_variable(varname)
    VAR=np.zeros_like(Z,dtype='float32')
    
    Nt=len(data['time'])
    Np=len(parcel['xh'])
    for p in np.arange(Np):
        if np.mod(p,2e5)==0: print(f"{p}/{len(parcel['xh'])}")
    
        #Get Indicies
        zs=Z[:,p]
        ys=Y[:,p]
        xs=X[:,p]
        ts = np.arange(Nt)  
    
        #Get Values
        vars = var_data[ts, zs, ys, xs]

        #Adding to Lagrangian Array
        VAR[:,p]=vars

        del vars
    del var_data
    return VAR

In [8]:
# print('working on W, QC, and QI\n')
# W=make_lagrangian_array('winterp'); check_memory() 
# QC=make_lagrangian_array('qc'); check_memory()
# QI = make_lagrangian_array('qi'); check_memory()


# print('working on QC+QI\n')
# import dask.array as da
# Nt=len(data['time'])
# QC = da.from_array(QC, chunks=(Nt, 'auto'))
# QI = da.from_array(QI, chunks=(Nt, 'auto'))
# QCQI=QC+QI
# QCQI=QCQI.compute(); check_memory()


print('working on all other variables\n')
QV=make_lagrangian_array('qv');check_memory()
TH=make_lagrangian_array('th');check_memory()
TH_E=make_lagrangian_array('th_e');check_memory()
BUOYANCY=make_lagrangian_array('buoyancy');check_memory()

print('working on HMC\n')
HMC=make_lagrangian_array('HMC');check_memory()

print('working on VMF_c\n')
VMF_c=make_lagrangian_array('VMF_c');check_memory()

print('working on VMF_g\n')
VMF_g=make_lagrangian_array('VMF_g');check_memory()

working on all other variables

0/1000000
200000/1000000
400000/1000000
600000/1000000
800000/1000000
Top 10 objects with highest memory usage
{'Z': '1064.0 MB', 'Y': '1064.0 MB', 'X': '1064.0 MB', 'QV': '532.0 MB', 'NamespaceMagics': '0.0 MB', 'Normalize': '0.0 MB', 'MaxNLocator': '0.0 MB', 'ScalarFormatter': '0.0 MB', 'times': '0.0 MB', 'open_file': '0.0 MB'}

3.724 GB in use overall
0/1000000
200000/1000000
400000/1000000
600000/1000000
800000/1000000
Top 10 objects with highest memory usage
{'Z': '1064.0 MB', 'Y': '1064.0 MB', 'X': '1064.0 MB', 'QV': '532.0 MB', 'TH': '532.0 MB', 'NamespaceMagics': '0.0 MB', 'Normalize': '0.0 MB', 'MaxNLocator': '0.0 MB', 'ScalarFormatter': '0.0 MB', 'times': '0.0 MB'}

4.256 GB in use overall
0/1000000
200000/1000000
400000/1000000
600000/1000000
800000/1000000
Top 10 objects with highest memory usage
{'Z': '1064.0 MB', 'Y': '1064.0 MB', 'X': '1064.0 MB', 'QV': '532.0 MB', 'TH': '532.0 MB', 'TH_E': '532.0 MB', 'NamespaceMagics': '0.0 MB', 'Normali

In [9]:
# Saving Data
##############
import h5py
dir2=dir+'Project_Algorithms/Lagrangian_Arrays/'
out_file=dir2+f'VARS_binary_array_{res}_{Np_str}_5min.h5'
with h5py.File(out_file, 'w') as f:
    # Save the array as a variable in the file
    # f.create_dataset('W', data=W) 
    # f.create_dataset('QCQI', data=QCQI)
    f.create_dataset('QV', data=QV)
    f.create_dataset('TH', data=TH)
    f.create_dataset('TH_E', data=TH_E)
    f.create_dataset('BUOYANCY', data=BUOYANCY)
    f.create_dataset('HMC', data=HMC)
    f.create_dataset('VMF_c', data=VMF_c)
    f.create_dataset('VMF_g', data=VMF_g)
check_memory()

Top 10 objects with highest memory usage
{'Z': '1064.0 MB', 'Y': '1064.0 MB', 'X': '1064.0 MB', 'QV': '532.0 MB', 'TH': '532.0 MB', 'TH_E': '532.0 MB', 'BUOYANCY': '532.0 MB', 'HMC': '532.0 MB', 'VMF_c': '532.0 MB', 'VMF_g': '532.0 MB'}

6.916 GB in use overall


In [ ]:
#########################################

In [8]:
# # Reading Back Data Later
# ##############
# import h5py
# dir2=dir+'Project_Algorithms/Lagrangian_Arrays/'
# with h5py.File(dir2+f'VARS_binary_array_{res}_{Np_str}_5min.h5', 'r') as f:
#     # Load the dataset by its name
#     QV = f['QV'][:]
#     TH = f['TH'][:]
#     TH_E = f['TH_E'][:]
#     BUOYANCY = f['BUOYANCY'][:]
#     HMC = f['HMC'][:]
#     VMF_C = f['VMF_C'][:]
# check_memory()

Top 10 objects with highest memory usage
{'Z': '1064.0 MB', 'Y': '1064.0 MB', 'X': '1064.0 MB', 'QV': '532.0 MB', 'TH': '532.0 MB', 'TH_E': '532.0 MB', 'BUOYANCY': '532.0 MB', 'HMC': '532.0 MB', 'NamespaceMagics': '0.0 MB', 'Normalize': '0.0 MB'}

5.852 GB in use overall
